# Estúdio de audiolivros

Um notebook só para todos os livros. **Escolha o livro na célula 0 e clique em `Ambiente de execução → Executar tudo`.** Ele narra só os capítulos que ainda não têm MP3 no seu Drive; os que já estão prontos são pulados.

**Antes:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`. A única coisa que você precisa fazer é autorizar o Google Drive quando aparecer a janela (célula 2).

| Livro | Obra | Tipo | Pasta no Drive |
|---|---|---|---|
| `narizinho` | Reinações de Narizinho (Monteiro Lobato) | pcd | `MyDrive/Audiolivros/narizinho/` |
| `sherlock` | Sherlock Holmes (Conan Doyle) | pcd | `MyDrive/Audiolivros/sherlock/` |
| `subsolo` | Memórias do Subsolo (Dostoiévski) | pessoal | `MyDrive/Audiolivros-pessoais/subsolo/` |

- **pcd:** `capNN_A_so_voz.mp3` e `capNN_B_com_ambiente.mp3` (vinheta e sonoplastia leve).
- **pessoal:** `capNN_guiada.mp3` (com comentários) e `capNN_limpa.mp3` (só o texto).
- No fim, a célula 8 põe cópias com nome legível em `<livro>/ouvir/` (bom para o celular) e escreve `situacao.md` com o que está pronto.

**Outro livro na mesma sessão:** troque o livro na célula 0 e clique de novo em *Executar tudo*. A instalação, o servidor e as licenças já prontos são pulados em segundos.

**Editar um roteiro sem mexer no git:** ponha os JSON em `<livro>/roteiros/` no Drive. Se essa pasta existir, o notebook usa ela.

**Licença:** a voz (OmniVoice) é de uso **não comercial** (CC-BY-NC). As vozes são sintéticas, sem clonar ninguém. Lições aprendidas: `LICOES.md`.

In [ ]:
# ── 0. Escolha o livro ──────────────────────────────────────────────────────
LIVRO = "narizinho"  #@param ["narizinho", "sherlock", "subsolo"] {allow-input: true}
CAPITULOS = ""  #@param {type:"string"}
REFAZER = False  #@param {type:"boolean"}
# CAPITULOS: vazio = só os que faltam; "todos"; ou uma lista como "4-10, 12".
# REFAZER: True narra de novo os escolhidos, mesmo que os MP3 já existam
# (trechos .wav que não mudaram são reaproveitados).
ESTUDIO_BRANCH = "main"

In [ ]:
# ── 1. Conferir a GPU ───────────────────────────────────────────────────────
import shutil, subprocess
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
else:
    raise SystemExit("Sem GPU. Vá em Ambiente de execução -> Alterar o tipo de ambiente "
                     "de execução -> GPU T4, e rode o notebook de novo do início.")

In [ ]:
# ── 2. Google Drive ─────────────────────────────────────────────────────────
# Autorize na janela que abrir (é o único passo manual).
import os
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
print("Drive pronto.")

In [ ]:
# ── 3. Instalar o VoiceStudio (primeira vez: ~5-8 min; depois, segundos) ────
import os, shutil, subprocess, sys

REPO_URL = "https://github.com/sdpaula36-stack/VoiceStudio"
REPO_DIR = "/content/VoiceStudio"
MARCA = "/content/.voicestudio_instalado"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

def run(cmd, *, cwd=None, what=""):
    shown = cmd if isinstance(cmd, str) else " ".join(cmd)
    print(f"\n$ {shown}")
    if subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode != 0:
        raise SystemExit(f"\nFALHOU: {what or shown}. Veja o erro acima; rodar de novo é seguro.")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    run(["git", "clone", "--depth", "1", "--branch", ESTUDIO_BRANCH, REPO_URL, REPO_DIR], what="git clone")
else:   # clone já existe (até de outro notebook): só traz a versão nova do estúdio
    run(["git", "-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin",
         f"+refs/heads/{ESTUDIO_BRANCH}:refs/remotes/origin/{ESTUDIO_BRANCH}"], what="git fetch")
    run(["git", "-C", REPO_DIR, "checkout", "-q", "-B", "estudio", f"origin/{ESTUDIO_BRANCH}"], what="git checkout")

if not os.path.exists(MARCA):
    run("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1", what="apt-get install")
    if not shutil.which("uv"):
        run([sys.executable, "-m", "pip", "install", "-q", "uv"], what="pip install uv")
    run(["uv", "pip", "install", "--system", "--no-cache",
         "--constraint", "deploy/torch-constraints.txt",
         "--override", "deploy/uv-overrides.txt", "."],
        cwd=REPO_DIR, what="instalação do backend")
    from huggingface_hub import snapshot_download
    print("Baixando o modelo de voz (alguns GB, só na primeira vez)...")
    snapshot_download("k2-fsa/OmniVoice")
    open(MARCA, "w").close()
print("Instalado.")

sys.path.insert(0, os.path.join(REPO_DIR, "notebooks", "estudio-audiolivros"))
import importlib, estudio as E
importlib.reload(E)

In [ ]:
# ── 4. Ligar o servidor do VoiceStudio ──────────────────────────────────────
import json, os, subprocess, sys, time, urllib.request

PORT = 3900
BASE = f"http://127.0.0.1:{PORT}"
LOG_PATH = "/content/voicestudio_backend.log"

def health():
    try:
        with urllib.request.urlopen(f"{BASE}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if not info:
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    env = os.environ.copy()
    env.update(OMNIVOICE_SERVER_MODE="1", PYTHONUNBUFFERED="1")
    env.setdefault("OMNIVOICE_DATA_DIR", "/content/omnivoice_data")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
         "--host", "127.0.0.1", "--port", str(PORT)],
        cwd="/content/VoiceStudio", env=env,
        stdout=open(LOG_PATH, "ab"), stderr=subprocess.STDOUT)
    deadline = time.time() + 300
    while time.time() < deadline and proc.poll() is None and not info:
        print(".", end="", flush=True)
        time.sleep(3)
        info = health()
    print()
    if not info:
        raise SystemExit("O servidor não ligou em 5 min. Final do log:\n"
                         + "".join(open(LOG_PATH, errors="replace").readlines()[-40:]))
print("Servidor ligado:", info)

In [ ]:
# ── 4.5. Aceitar as licenças dos modelos (obrigatório antes de gerar) ───────
# Sem este passo o /generate falha. Rodar esta célula aceita as licenças em seu nome:
# OmniVoice é CC-BY-NC (uso NÃO comercial: o áudio não pode ser vendido).
import requests
BASE = "http://127.0.0.1:3900"

for repo in ("k2-fsa/OmniVoice", "eustlb/higgs-audio-v2-tokenizer"):
    st = requests.get(f"{BASE}/models/licenses/acceptance/{repo}", timeout=30).json()
    if st.get("accepted"):
        print(f"[ok] {repo} já aceito")
        continue
    r = requests.post(
        f"{BASE}/models/licenses/accept",
        json={"repo_id": repo, "fingerprint": st["fingerprint"], "accepted": True},
        timeout=30,
    )
    r.raise_for_status()
    print(f"[aceito] {repo} ({st.get('license')})")

In [ ]:
# ── 5. Carregar e conferir os roteiros do livro ─────────────────────────────
CFG = E.livro(LIVRO)
TIPO = CFG["tipo"]
PASTA_LIVRO = E.pasta_drive(CFG)
os.makedirs(PASTA_LIVRO, exist_ok=True)
print(f"{CFG['titulo']} ({TIPO}) -> {PASTA_LIVRO}")

ROTEIROS = E.carregar_roteiros(LIVRO, REPO_DIR, PASTA_LIVRO)
if E.relatorio(ROTEIROS, TIPO):
    raise SystemExit("O roteiro tem erros (acima). Nada foi narrado.")
VOZES, CAPITULOS_DO_LIVRO = E.juntar(ROTEIROS)
FILA = E.escolher(CAPITULOS, CAPITULOS_DO_LIVRO, PASTA_LIVRO, TIPO, REFAZER)
prontos = [n for n in CAPITULOS_DO_LIVRO if E.pronto(PASTA_LIVRO, TIPO, n)]
print(f"\nJá prontos: {len(prontos)} de {len(CAPITULOS_DO_LIVRO)}. Vou narrar agora: {FILA or 'nada'}")

In [ ]:
# ── 6. Vozes (as que já existem no Drive são reaproveitadas) ────────────────
NARRADOR = E.Narrador(BASE, PASTA_LIVRO, TIPO, VOZES, CAPITULOS_DO_LIVRO, LOG_PATH)
NARRADOR.criar_vozes()
# Voz ruim ou com sotaque de Portugal: troque a "semente" no roteiro, apague o .wav
# dela em <livro>/vozes/ e rode de novo.

In [ ]:
# ── 7. Narrar e montar os capítulos da fila ─────────────────────────────────
import time
for n in FILA:
    t0 = time.time()
    NARRADOR.capitulo(n)
    print(f"  ({(time.time() - t0) / 60:.1f} min de GPU)")
print("\nNarração concluída." if FILA else "Nada para narrar: todos os capítulos escolhidos já estão prontos.")
# Um trecho ruim: apague o .wav dele em <livro>/capNN/trechos/ e rode com CAPITULOS="NN" e REFAZER=True.

In [ ]:
# ── 8. Organizar no Drive (pasta ouvir/, playlists e situacao.md) ───────────
E.organizar(PASTA_LIVRO, TIPO, CAPITULOS_DO_LIVRO, CFG["titulo"])
print("\nPronto! Tudo em", PASTA_LIVRO)